# 토큰화와 품사 태깅

## 01. 토큰화의 정의와 처리 단위

### 토큰화란 무엇인가

**토큰화(Tokenization)** 는 원문 문자열을 분석 목적에 맞는 경계로 나누어 **순서가 있는 토큰 목록**을 만드는 과정이다.
토큰은 이후 정수 ID나 벡터로 변환되는 최소 입력 단위이며, 토큰화 자체는 아직 정수 변환이나 벡터화를 수행하지 않는다.

- **경계 규칙:** 공백만 사용할 수도 있고, 언어별 문법·정규식·학습된 어휘 사전을 사용할 수도 있다.
- **필요한 이유:** 컴퓨터가 문장의 구조를 계산하려면 먼저 어느 구간을 하나의 처리 단위로 볼지 결정해야 하기 때문이다.

토큰화 뒤에는 보통 `토큰 목록 → 정수 ID → 패딩 또는 벡터 → 모델 입력` 순서가 이어진다. 따라서 토큰화와 정수 인코딩을 같은 작업으로 보면 안 된다.

### 토큰 단위별 경계와 차이

#### 문장 토큰화

문서에서 문장 경계를 찾아 문장 문자열 목록을 만든다. `오늘은 맑다. 산책을 간다.`를 `['오늘은 맑다.', '산책을 간다.']`로 나누는 방식이다. 약어·따옴표·물음표 같은 문장 종결 예외를 처리해야 한다.

#### 단어 토큰화

문장 안에서 단어와 구두점의 경계를 찾는다. 영어에서는 `Don't stop.`이 토크나이저에 따라 `['Do', "n't", 'stop', '.']`처럼 나뉠 수 있다. 한국어의 공백 단위인 어절은 조사와 어미가 붙어 있으므로 어절 토큰과 형태소 토큰의 경계는 서로 다르다.

#### 형태소 분석

형태소는 의미나 문법 기능을 가진 가장 작은 단위이다. 예를 들어 `사과를 먹었다`는 분석기에 따라 `사과/NNG`, `를/JKO`, `먹/VV`, `었/EP`, `다/EF`처럼 어근·조사·어미로 나뉜다. 형태소 분석은 언어의 문법과 품사 사전을 사용하며, 단순한 공백 분리보다 언어 의존성이 크다.

#### 서브워드 토큰화

학습된 어휘 사전을 기준으로 단어를 더 작은 문자열 조각으로 나눈다. BERT WordPiece는 `unhappiness`를 `un`, `##ha`, `##pp`, `##iness`처럼 나눌 수 있다. 서브워드는 OOV를 줄이기 위한 통계적 문자열 단위이므로 형태소와 항상 일치하지 않는다.

#### 문자 토큰화

문자열을 문자 하나씩 나눈다. `NLP`는 `['N', 'L', 'P']`가 된다. 미등록 문자는 적지만 시퀀스가 길어지고 단어 수준의 의미를 더 긴 범위에서 학습해야 한다.

### 핵심 용어

- **토큰(Token):** 토큰화 결과로 얻은 하나의 처리 단위이다.
- **어휘 사전(Vocabulary):** 모델이나 벡터화기가 알고 있는 고유 토큰의 집합이다.
- **토큰 ID(Token ID):** 어휘 사전에서 각 토큰에 부여한 정수 인덱스이다.
- **특수 토큰(Special Token):** 문장의 시작·끝·패딩·미등록어처럼 모델 제어에 사용하는 토큰이다.
- **OOV(Out Of Vocabulary):** 현재 어휘 사전에 등록되지 않은 토큰이다.

같은 원문도 분석 목적에 따라 적절한 경계가 다르다. 검색에서는 제품명 전체를 보존할 수 있고, 형태소 분석에서는 조사와 어미를 분리할 수 있으며, BERT 입력에서는 해당 모델과 함께 배포된 서브워드 규칙을 사용해야 한다.


## 02. 실습 환경과 도구

### 필요한 패키지와 spaCy 영어 모델 설치

`%pip install`은 현재 Jupyter 커널 환경에 NLTK, KSS, Transformers와 spaCy를 설치한다. spaCy 영어 모델 `en_core_web_sm`은 패키지와 별도로 배포되므로 `python -m spacy download en_core_web_sm`으로 내려받는다.

설치가 끝나면 다음 셀에서 각 라이브러리와 영어 모델을 직접 불러온다.


In [1]:
from fastjsonschema.generator import enforce_list
%pip install -q nltk kss transformers spacy
!python -m spacy download en_core_web_sm


Note: you may need to restart the kernel to use updated packages.
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 12.8/12.8 MB 72.9 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


### 사용할 라이브러리와 언어 모델 불러오기

각 라이브러리는 서로 다른 토큰화와 후속 분석 기능을 담당한다.

- **NLTK:** 영어 문장·단어 토큰화, VADER 감성 분석과 품사 태깅에 사용한다.
- **KSS:** 한국어 종결 표현과 인용 구조를 고려한 문장 분리에 사용한다.
- **Transformers의 `AutoTokenizer`:** BERT WordPiece 토크나이저를 불러오는 데 사용한다.
- **spaCy:** 영어 토큰화와 품사 태깅 파이프라인에 사용한다.
- **`en_core_web_sm`:** spaCy가 사용하는 영어 품사 태깅·표제어 추출 모델이다.


In [2]:
import nltk
import kss
from transformers import AutoTokenizer
import spacy
import en_core_web_sm


### NLTK 데이터 리소스 다운로드

NLTK의 파이썬 패키지와 토크나이저·감성 사전·품사 태거 데이터는 별도로 배포된다.
토큰화, VADER 감성 분석과 영어 품사 태깅에 필요한 다섯 리소스를 `nltk.download()`로 바로 내려받는다.


In [3]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("vader_lexicon")
nltk.download("averaged_perceptron_tagger")
nltk.download("averaged_perceptron_tagger_eng")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\playdata2\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\playdata2\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\playdata2\AppData\Roaming\nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\playdata2\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\playdata2\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger_eng.zip.


True

## 03. 같은 원문을 문장·단어·문자 단위로 나누기

토큰화 단위가 작아질수록 토큰 수와 시퀀스 길이는 늘어나지만 더 세밀한 문자열 정보를 보존할 수 있다. 문장 단위는 문서 구조, 단어 단위는 어휘와 구두점, 문자 단위는 공백을 포함한 개별 문자를 토큰으로 다룬다.

`sent_tokenize()`는 문장 문자열 목록을, `word_tokenize()`는 단어와 구두점 목록을 반환하며 `list(text)`는 문자열을 문자 목록으로 바꾼다. 결과에서는 단위별 토큰 수와 공백·구두점이 남는 방식을 비교한다.

단어 토큰화 결과는 형태소 분석 결과가 아니다. `real-world`가 보존되거나 분리되는 방식은 NLTK의 영어 토큰 경계 규칙에 따른다.


In [4]:
from nltk.tokenize import sent_tokenize, word_tokenize

sample_text = (
    "NLP is fascinating. "
    "It has many applications in real-world scenarios."
)

print(sample_text)
print(type(sample_text))

# 토근화
sent_tokens = sent_tokenize(sample_text) # 문장 토큰화
word_tokens = word_tokenize(sample_text) # 단어 토큰화
character_tokens = list(sample_text) # 문자 토큰화

print('문장 토큰 : ', len(sent_tokens), sent_tokens)
print('단어 토큰 : ', len(word_tokens), word_tokens)
print('문자 토큰 : ', len(character_tokens), character_tokens)

NLP is fascinating. It has many applications in real-world scenarios.
<class 'str'>
문장 토큰 :  2 ['NLP is fascinating.', 'It has many applications in real-world scenarios.']
단어 토큰 :  12 ['NLP', 'is', 'fascinating', '.', 'It', 'has', 'many', 'applications', 'in', 'real-world', 'scenarios', '.']
문자 토큰 :  69 ['N', 'L', 'P', ' ', 'i', 's', ' ', 'f', 'a', 's', 'c', 'i', 'n', 'a', 't', 'i', 'n', 'g', '.', ' ', 'I', 't', ' ', 'h', 'a', 's', ' ', 'm', 'a', 'n', 'y', ' ', 'a', 'p', 'p', 'l', 'i', 'c', 'a', 't', 'i', 'o', 'n', 's', ' ', 'i', 'n', ' ', 'r', 'e', 'a', 'l', '-', 'w', 'o', 'r', 'l', 'd', ' ', 's', 'c', 'e', 'n', 'a', 'r', 'i', 'o', 's', '.']


### 서브워드 토큰화: BERT WordPiece

서브워드 토큰화는 학습된 어휘 사전에 자주 등장하는 문자열 조각을 토큰으로 사용한다. 단어 전체가 어휘에 없어도 여러 조각으로 표현할 수 있어 OOV를 줄일 수 있다. WordPiece 조각은 통계적으로 학습된 문자열 단위이므로 형태소나 의미 단위와 항상 일치하지 않는다.

> **BERT(Bidirectional Encoder Representations from Transformers):** Transformer 인코더로 양방향 문맥을 학습한 사전학습 언어 모델이다.

`AutoTokenizer.from_pretrained("bert-base-uncased")`는 모델 이름에 맞는 토크나이저 파일을 내려받거나 기존 캐시에서 불러온다. `tokenize()`는 WordPiece 문자열 목록을 만들고 `convert_tokens_to_ids()`는 각 조각을 어휘 사전의 정수 ID로 변환한다.

BERT WordPiece에서 `##`는 해당 조각이 단어의 시작이 아니라 앞 조각에 이어지는 부분임을 뜻한다. `[CLS]`는 입력의 시작 위치에 두는 분류용 특수 토큰이고, `[SEP]`는 한 문장 또는 문장 쌍의 끝 경계를 표시한다. 토큰과 ID의 개수, `##`의 위치와 전체 입력 앞뒤에 추가된 특수 토큰을 함께 확인한다.


In [10]:
from transformers import AutoTokenizer

# bert-base-uncased : 영어 대소문자를 구분하지 않는 Bert 베이스 모델의 토크나이저 이름
model_name = 'bert-base-uncased'

# 사전 학습된 토크나이저 모델 'bert-base-uncased' 다운 받아서 불러오기
tokenizer = AutoTokenizer.from_pretrained(model_name)

subword_text = "unhappiness and reusable tokenizers"

subword_tokens = tokenizer.tokenize(subword_text)

# bert_base-uncased의 고정 어휘 사전에서 문자열 조각들이 어디 있었는지 인덱스 표시
subword_ids = tokenizer.convert_tokens_to_ids(subword_tokens)

# add_special_tokens=True : 토큰 목록 앞, 뒤에 [CLS] [SEP] 추가
# [CLS] : 문장의 시작(입력의 시작 위치)
# [SEP] : 문장의 끝 (경계)
encoded = tokenizer(subword_text, add_special_tokens=True)
encoded_ids = encoded['input_ids']
encoded_tokens = tokenizer.convert_ids_to_tokens(encoded_ids)


print("원본:" , subword_text)
print("서브워드토큰:" , subword_tokens)
print("서브워드 id:" , subword_ids)
print("특수 토큰 포함: ", encoded_tokens)



원본: unhappiness and reusable tokenizers
서브워드토큰: ['un', '##ha', '##pp', '##iness', 'and', 're', '##usa', '##ble', 'token', '##izer', '##s']
서브워드 id: [4895, 3270, 9397, 9961, 1998, 2128, 10383, 3468, 19204, 17629, 2015]
특수 토큰 포함:  ['[CLS]', 'un', '##ha', '##pp', '##iness', 'and', 're', '##usa', '##ble', 'token', '##izer', '##s', '[SEP]']


### 04. 영어 단어 토큰화 규칙 비교

아포스트로피가 있는 축약형과 하이픈 복합어는 토크나이저의 규칙에 따라 서로 다르게 나뉜다. `re.findall()`은 현재 과제에 필요한 경계를 직접 정의하지만 예외도 직접 관리해야 한다.

`WordPunctTokenizer`는 구두점을 적극적으로 분리하고, `word_tokenize()`는 영어 축약형을 고려해 `Don't`를 `Do`, `n't`로 나눈다. `well-being`과 `self-care`가 세 방식에서 어떻게 달라지는지 비교하면 과업에 맞는 토큰 경계를 선택할 수 있다. 정규 표현식의 상세 문법은 뒤의 정규 표현식 노트북에서 다룬다.


In [11]:
import re
from nltk.tokenize import WordPunctTokenizer, word_tokenize

# 첫 대안은 영문 단어를 찾고 (?:[-'][A-Za-z]+)*는 단어 내부 아포스트로피·하이픈 부분을 함께 잡는다.
# 두 번째 대안 [^\w\s]는 단어 문자나 공백이 아닌 나머지 구두점 한 글자를 별도 토큰으로 잡는다.

comparison_text = "Don't hesitate to use well-being practices for self-care."

regex_pattern = r"[A-Za-z]+(?:[-'][A-Za-z]+)*|[^\w\s]"
regex_tokens = re.findall(regex_pattern, comparison_text)

word_punct_tokens = WordPunctTokenizer().tokenize(comparison_text)
nltk_word_tokens = word_tokenize(comparison_text)

print(f"정규식 ({len(regex_tokens)}개)            :", regex_tokens)
print(f"WordPunctTokenizer ({len(word_punct_tokens)}개):", word_punct_tokens)
print(f"word_tokenize ({len(nltk_word_tokens)}개)      :", nltk_word_tokens)

정규식 (9개)            : ["Don't", 'hesitate', 'to', 'use', 'well-being', 'practices', 'for', 'self-care', '.']
WordPunctTokenizer (15개): ['Don', "'", 't', 'hesitate', 'to', 'use', 'well', '-', 'being', 'practices', 'for', 'self', '-', 'care', '.']
word_tokenize (10개)      : ['Do', "n't", 'hesitate', 'to', 'use', 'well-being', 'practices', 'for', 'self-care', '.']


### 도메인 표현을 보존하는 토큰 경계

가격·날짜·제품명·질병명은 구두점을 포함해야 하나의 값이 될 수 있다. 분석 과제에서 문자열 전체가 하나의 특성이어야 한다면 기본 토크나이저보다 도메인 규칙을 우선한다.

`WordPunctTokenizer`는 `COVID-19`, `€100.50`, `2023/10/30`의 구두점을 경계로 나누지만, 사용자 정규식은 세 표현을 각각 하나의 토큰으로 보존할 수 있다. 정규식의 대안은 왼쪽부터 검사하므로 구체적인 가격·날짜 패턴을 일반 숫자·단어 패턴보다 먼저 배치해야 한다.


In [12]:
from nltk.tokenize import WordPunctTokenizer

# 질병명·가격·날짜처럼 구두점까지 포함해야 하나의 값이 되는 표현을 넣은 도메인 문장
domain_text = "COVID-19 costs €100.50 on 2023/10/30, according to NASA."

# 비교 기준은 구두점을 경계로 분리하므로 도메인 표현이 여러 토큰으로 나뉠 수 있다.
baseline_tokens = WordPunctTokenizer().tokenize(domain_text)

# 정규식 대안을 왼쪽부터 검사하므로 구체적인 가격·날짜·하이픈 표현을 일반 숫자·단어보다 먼저 둔다.
# 마지막 [^\w\s]는 앞 패턴에 해당하지 않은 쉼표와 마침표 같은 구두점을 별도 토큰으로 남긴다.
domain_pattern = (
    r"€\d+(?:\.\d+)?"
    r"|\d{4}/\d{1,2}/\d{1,2}"
    r"|[A-Za-z]+(?:-[A-Za-z0-9]+)*"
    r"|\d+(?:\.\d+)?"
    r"|[^\w\s]"
)

domain_tokens = re.findall(domain_pattern, domain_text)

print(f"구두점 중심 기준 ({len(baseline_tokens)}개):", baseline_tokens)
print(f"도메인 규칙 ({len(domain_tokens)}개)      :", domain_tokens)

구두점 중심 기준 (19개): ['COVID', '-', '19', 'costs', '€', '100', '.', '50', 'on', '2023', '/', '10', '/', '30', ',', 'according', 'to', 'NASA', '.']
도메인 규칙 (10개)      : ['COVID-19', 'costs', '€100.50', 'on', '2023/10/30', ',', 'according', 'to', 'NASA', '.']


## 05. 영어 문장 토큰화 (Punkt 모델)

문장 경계는 모든 마침표에서 단순히 나눌 수 없다. 약어의 마침표, 물음표·느낌표, 따옴표 안의 종결 표현을 함께 고려해야 한다.

NLTK의 `sent_tokenize()`가 사용하는 Punkt는 구두점뿐 아니라 학습된 약어와 문장 경계 정보를 활용한다. 따라서 결과에서는 `Dr.`과 같은 약어 뒤가 문장 경계로 잘못 처리되지 않고 `Dr. Zhang works at NASA.`가 하나의 문장으로 유지되는지 확인해야 한다.


In [14]:
from nltk.tokenize import sent_tokenize

english_text = (
    "Hello world! Dr. Zhang works at NASA. "
    "How are you today? I'm learning sentence tokenization."
)

# sent_tokenize() : Punkt 규칙을 이용해서 원문을 문장 단위 토큰화
english_sentences = sent_tokenize(english_text)

for index, sentences in enumerate(english_sentences, start = 1):
    print(f'{index} : {sentences}')

print("문장 수 : ", len(english_sentences))

1 : Hello world!
2 : Dr. Zhang works at NASA.
3 : How are you today?
4 : I'm learning sentence tokenization.
문장 수 :  4


### 한국어 문장 토큰화: 영어 Punkt와 KSS 비교

문장 경계를 판단하는 기준은 언어마다 다르다. 영어는 약어와 구두점이 중요한 반면, 한국어는 종결 어미와 인용 표현이 문장 경계 판단에 영향을 준다.

NLTK의 기본 Punkt 모델은 영어를 기준으로 만들어졌기 때문에 한국어의 종결 어미와 인용 구조를 충분히 반영하지 못할 수 있다. KSS는 한국어 문맥과 종결 표현을 고려해 문장을 분리한다.

결과에서 영어 Punkt는 인용문을 둘로 나누어 5개 문장을 만들고, KSS는 `"내일도 비가 올까?"라고 친구가 물었다.`를 하나로 유지해 4개 문장을 만든다. 문장 수뿐 아니라 실제로 분리된 문자열을 읽어야 잘못된 경계를 판단할 수 있다.


In [19]:
from nltk.tokenize import sent_tokenize
import kss

korean_text = (
    '오늘은 비가 온다. 우산을 챙겼다. '
    '"내일도 비가 올까?"라고 친구가 물었다. 그래도 우리는 산책을 갔다.'
)

# NLTK의 Punkt를 이용해서 한국어 문장 토큰화
nltk_korean_sentences = sent_tokenize(korean_text)

print("[NLTK]")
for index, sentences in enumerate(nltk_korean_sentences, start = 1):
    print(f'{index} : {sentences}')
print("문장 수 : ", len(nltk_korean_sentences))

# KSS의 종결어미, 인용구문을 이용해서 한국어 문장 토큰화
kss_korean_sentences = kss.split_sentences(korean_text)

print("[KSS]")
for index, sentences in enumerate(kss_korean_sentences, start = 1):
    print(f'{index} : {sentences}')
print("문장 수 : ", len(kss_korean_sentences))

[NLTK]
1 : 오늘은 비가 온다.
2 : 우산을 챙겼다.
3 : "내일도 비가 올까?
4 : "라고 친구가 물었다.
5 : 그래도 우리는 산책을 갔다.
문장 수 :  5
[KSS]
1 : 오늘은 비가 온다.
2 : 우산을 챙겼다.
3 : "내일도 비가 올까?"라고 친구가 물었다.
4 : 그래도 우리는 산책을 갔다.
문장 수 :  4


## 06. 토큰화가 후속 감성 분석에 미치는 영향

토큰 경계가 달라지면 감성 분석기가 인식하는 단어와 감성 규칙도 달라질 수 있다. 특히 `don't` 같은 부정 표현이나 느낌표·대문자 강조가 분리되면 원문에 있던 감성 정보가 손실될 수 있다.

> **VADER(Valence Aware Dictionary and sEntiment Reasoner):** 영어 소셜 텍스트의 감성 강도를 감성 사전과 규칙으로 계산하는 분석기이다.

VADER는 감성 단어뿐 아니라 부정 표현, 대문자, 구두점과 강조 표현을 함께 해석한다. 따라서 일반적으로 토큰 목록을 공백으로 다시 결합한 문자열보다 원문 문자열을 입력하는 것이 적절하다.

`polarity_scores()`는 부정·중립·긍정 비율과 -1에서 1 사이의 종합 점수 `compound`를 반환한다. 원문의 감성 점수를 기준으로 두고, 서로 다른 토큰화 결과를 재결합했을 때 `compound`의 방향과 변화량이 어떻게 달라지는지 비교한다.


In [16]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.tokenize import WordPunctTokenizer, word_tokenize

sentiment_analyzer = SentimentIntensityAnalyzer()
sentiment_text = "I don't like the way she talks about mental well-being."

word_punct_text = " ".join(WordPunctTokenizer().tokenize(sentiment_text))
nltk_word_text = " ".join(word_tokenize(sentiment_text))

sentiment_inputs = {
    "원문 기준선": sentiment_text,
    "WordPunct 재결합": word_punct_text,
    "word_tokenize 재결합": nltk_word_text,
}
sentiment_results = {}

for label, candidate_text in sentiment_inputs.items():
    scores = sentiment_analyzer.polarity_scores(candidate_text)
    sentiment_results[label] = scores
    print(f"[{label}] {candidate_text}")
    print(f"  compound={scores['compound']:.4f}, 전체 점수={scores}")

baseline_compound = sentiment_results["원문 기준선"]["compound"]
for label in ("WordPunct 재결합", "word_tokenize 재결합"):
    delta = sentiment_results[label]["compound"] - baseline_compound
    print(f"{label}의 원문 대비 compound 변화: {delta:+.4f}")

[원문 기준선] I don't like the way she talks about mental well-being.
  compound=-0.2755, 전체 점수={'neg': 0.209, 'neu': 0.791, 'pos': 0.0, 'compound': -0.2755}
[WordPunct 재결합] I don ' t like the way she talks about mental well - being .
  compound=0.5574, 전체 점수={'neg': 0.0, 'neu': 0.635, 'pos': 0.365, 'compound': 0.5574}
[word_tokenize 재결합] I do n't like the way she talks about mental well-being .
  compound=-0.2755, 전체 점수={'neg': 0.19, 'neu': 0.81, 'pos': 0.0, 'compound': -0.2755}
WordPunct 재결합의 원문 대비 compound 변화: +0.8329
word_tokenize 재결합의 원문 대비 compound 변화: +0.0000


## 07. 토큰화 이후의 품사 태깅

품사 태깅은 토큰화된 각 단어에 명사·동사·형용사 같은 문법적 역할을 예측해 붙이는 후속 단계이다. 같은 표면형 단어도 문맥에 따라 다른 품사가 될 수 있으므로 토큰 목록과 주변 문맥이 함께 필요하다.

> **POS(Part Of Speech):** 문장에서 단어가 수행하는 문법적 역할을 나타내는 품사 정보이다.

`word_tokenize()`가 만든 토큰 목록을 `pos_tag()`에 전달하면 같은 순서의 `(토큰, Penn Treebank 태그)` 튜플 목록이 반환된다. 태그는 사전의 고정 정답이 아니라 학습된 모델의 예측 결과이므로 문맥과 모델 버전에 따라 달라질 수 있다.

자주 사용하는 Penn Treebank 태그는 다음과 같다.

| 태그 | 의미 | 예시 |
|---|---|---|
| `NN`, `NNS` | 단수·복수 일반 명사 | `book`, `books` |
| `NNP`, `NNPS` | 단수·복수 고유 명사 | `Alice`, `Smiths` |
| `VB`, `VBP`, `VBZ` | 동사 원형·현재형 | `run`, `run`, `runs` |
| `VBD`, `VBG`, `VBN` | 과거·현재분사·과거분사 | `ran`, `running`, `eaten` |
| `JJ`, `RB` | 형용사·부사 | `happy`, `quickly` |
| `DT`, `PRP`, `IN` | 한정사·대명사·전치사 | `the`, `she`, `in` |


In [18]:
from nltk import pos_tag, word_tokenize

pos_text = 'I love hamburgers!'

pos_tokens = word_tokenize(pos_text) # 토큰화

pos_tags = pos_tag(pos_tokens) # 품사 태그

print("토큰: ", pos_tokens)
print("품사 태그 : ", pos_tags)
print(f'토큰 수/태그 수 : {len(pos_tokens)} / {len(pos_tags)}')

토큰:  ['I', 'love', 'hamburgers', '!']
품사 태그 :  [('I', 'PRP'), ('love', 'VBP'), ('hamburgers', 'NNS'), ('!', '.')]
토큰 수/태그 수 : 4 / 4


### 같은 단어의 문맥별 품사 비교

`record`는 `They record music.`에서는 동사이고 `This is a record.`에서는 명사이다. `pos_tag()` 결과에서 첫 문장의 `record`는 `VBP`, 둘째 문장의 `record`는 `NN`으로 구분된다.

철자가 같은 단어도 주변 단어와 문장 구조에 따라 품사가 달라진다. 따라서 품사 태깅 결과는 개별 단어만 보지 않고 원문 문맥과 함께 해석해야 한다.


In [23]:
record_examples = [
    "They record music.",
    "This is a record."
]

for sentence in record_examples:

    # 문장 -> 토큰화 -> 품사 태그 붙이다
    tagged = pos_tag(word_tokenize(sentence))

    print(tagged)

    # next() : 다음 요소 1개 꺼내기
    record_tag = next(
       tag for token, tag in tagged if token.lower() == 'record'
    )

    print(f'{sentence:20} -> record/{record_tag} | 전체 = {tagged}')

[('They', 'PRP'), ('record', 'VBP'), ('music', 'NN'), ('.', '.')]
They record music.   -> record/VBP | 전체 = [('They', 'PRP'), ('record', 'VBP'), ('music', 'NN'), ('.', '.')]
[('This', 'DT'), ('is', 'VBZ'), ('a', 'DT'), ('record', 'NN'), ('.', '.')]
This is a record.    -> record/NN | 전체 = [('This', 'DT'), ('is', 'VBZ'), ('a', 'DT'), ('record', 'NN'), ('.', '.')]


### spaCy 품사 태깅과 태그 체계 비교

spaCy는 토큰화, 품사 태깅과 개체명 인식 등을 하나의 파이프라인으로 제공하는 자연어 처리 라이브러리이다. `en_core_web_sm`은 영어 분석에 사용하는 소형 학습 모델이며 spaCy 패키지와 별도로 설치한다.

`spacy.load("en_core_web_sm")`은 영어 파이프라인을 불러오고 `nlp(text)`는 토큰과 분석 결과가 담긴 `Doc` 객체를 반환한다. `token.pos_`는 `NOUN`, `VERB`처럼 언어 간에 공통으로 사용하는 Universal POS의 큰 범주이고, `token.tag_`는 `NN`, `VBZ`처럼 영어 문법을 더 세밀하게 구분한 태그이다.

NLTK의 Penn Treebank 태그와 비교할 때는 spaCy의 `tag_`가 비슷한 세밀도를 가진다. `pos_`와 Penn Treebank 태그는 분류 수준이 다르므로 이름만 보고 일대일로 대응시키면 안 된다.


In [25]:
import spacy

# spacy 제공 en_core_web_sm 파이프라인 불러오기
# -> 영어 토크나이저, 품사 태거(tagger) 등이 포함되어 있다.
nlp = spacy.load("en_core_web_sm")

spacy_text = 'Time flies like an arrow!'

# 원문을 토큰화하고, 품사를 예측해 분석 결과를 가진 Doc 객체를 반환함
doc = nlp(spacy_text)

tagged_tokens = [(token.text, token.pos_, token.tag_ )for token in doc]

print(tagged_tokens)

[('Time', 'NOUN', 'NN'), ('flies', 'VERB', 'VBZ'), ('like', 'ADP', 'IN'), ('an', 'DET', 'DT'), ('arrow', 'NOUN', 'NN'), ('!', 'PUNCT', '.')]


## 08. 핵심 정리와 확인 질문

### 토큰화 단위 선택 기준

- **문장 토큰화:** 문서에서 문장 경계를 찾으며 약어·인용문·종결 부호를 확인한다.
- **단어 토큰화:** 단어와 구두점 경계를 찾으며 축약형·하이픈·도메인 표현의 보존 여부를 확인한다.
- **형태소 분석:** 어근·조사·어미처럼 문법 기능을 가진 단위로 나누며 언어별 분석기가 필요하다.
- **서브워드 토큰화:** 학습된 어휘 조각으로 단어를 나누어 OOV를 줄이며 형태소와 같다고 해석하지 않는다.
- **문자 토큰화:** 문자 단위로 나누어 OOV를 줄이지만 시퀀스가 길어진다.

### 도구와 결과 해석

| 목적 | 우선 고려할 방법 | 결과에서 확인할 내용 |
|---|---|---|
| 영어 문장 분리 | NLTK `sent_tokenize` | 약어와 종결 부호 경계 |
| 한국어 문장 분리 | KSS | 인용 표현, 종결 어미와 조사 |
| 영어 단어 분리 | NLTK 또는 도메인 정규식 | 축약형·하이픈·가격·날짜 보존 여부 |
| BERT 입력 생성 | 모델과 함께 제공되는 토크나이저 | `##`, 토큰 ID, `[CLS]`, `[SEP]` |
| 영어 품사 태깅 | NLTK 또는 spaCy | 태그 체계와 문맥별 예측 차이 |

토큰화는 문자열을 토큰 목록으로 만드는 단계이고, 정수 인코딩은 토큰을 ID로 바꾸는 단계이며, 품사 태깅은 토큰에 문법 정보를 붙이는 후속 단계이다. 세 작업의 입력과 출력을 구분해야 전체 전처리 흐름을 정확히 설명할 수 있다.

### 확인 질문

1. 어절 토큰화와 형태소 분석의 경계가 다른 이유는 무엇인가?
2. 문자 토큰화는 OOV가 적지만 시퀀스가 길어지는 이유가 무엇인가?
3. `don't`를 `don`, `'`, `t`로 나누면 감성 분석 결과가 달라질 수 있는 이유는 무엇인가?
4. BERT의 WordPiece 조각을 형태소와 같은 개념으로 보면 안 되는 이유는 무엇인가?
5. NLTK와 spaCy의 품사 태그를 비교할 때 태그 체계를 먼저 확인해야 하는 이유는 무엇인가?

다음 노트북에서는 토큰 경계를 정한 뒤 대소문자, 특수문자와 표기 변형을 정제·정규화하는 방법을 학습한다.